In [19]:
import pandas as pd

df = pd.read_json(
    "../data/chunk/vi-wiki.jsonl",
    lines=True,
    nrows=100
)

In [24]:
print(df.iloc[1]["text"])

Tiếng Việt, cũng gọi là tiếng Việt Nam hay Việt ngữ là ngôn ngữ của người Việt và là ngôn ngữ chính thức tại Việt Nam. Đây là tiếng mẹ đẻ của khoảng 85% dân cư Việt Nam cùng với hơn 4 triệu người Việt kiều. Tiếng Việt còn là ngôn ngữ thứ hai của các dân tộc thiểu số tại Việt Nam và là ngôn ngữ dân tộc thiểu số được công nhận tại Cộng hòa Séc. Dựa trên từ vựng cơ bản, tiếng Việt được phân loại là một ngôn ngữ thuộc ngữ hệ Nam Á. Tiếng Việt là ngôn ngữ có nhiều người nói nhất trong ngữ hệ này (nhiều hơn tổng số người nói của tất cả các ngôn ngữ còn lại trong ngữ hệ). Vì Việt Nam thuộc Vùng văn hoá Đông Á, tiếng Việt cũng chịu nhiều ảnh hưởng về từ tiếng Hán, do vậy là ngôn ngữ có ít điểm tương đồng nhất với các ngôn ngữ khác trong ngữ hệ Nam Á. Lịch sử. Theo A. G. Haudricourt giải thích từ năm 1954, nhóm ngôn ngữ Việt-Mường ở thời kỳ khoảng đầu Công nguyên là những ngôn ngữ hay phương ngữ không thanh điệu. Về sau, qua quá trình giao thoa với Hoa ngữ và nhất là với các ngữ thuộc ngữ hệ Ta

In [8]:
import pandas as pd

df = pd.read_csv("cleaned.csv", nrows=100)
print(df.head(50)["title"])

0                      Internet Society
1                            Tiếng Việt
2                                  Ohio
3                            California
4                             Thụy Điển
5                                   W3C
6      Bộ Kế hoạch và Đầu tư (Việt Nam)
7                                Hoa Kỳ
8                              Cao Bằng
9                                  Iraq
10                            Campuchia
11                                 VIQR
12               Sacramento, California
13                          Los Angeles
14                        San Francisco
15                            San Diego
16                    Người Mỹ gốc Việt
17         Giấy phép Tài liệu Tự do GNU
18                       Lý Thường Kiệt
19                        Hồ Biểu Chánh
20    Thảo luận MediaWiki:Contributions
21                       Bắc Kạn (tỉnh)
22                             Lạng Sơn
23                                    A
24                                    B


In [6]:
import re

import mwparserfromhell
import pandas as pd
from tqdm import tqdm

tqdm.pandas(desc="Cleaning Wiki Text")


BAD_PREFIXES = (
    "Wikipedia:",
    "Tập tin:",
    "Bản mẫu:",
    "Thể loại:",
    "Thảo luận:",
    "Thảo luận Wikipedia:",
    "Thảo luận Thành viên:",
    "Thảo luận Tập tin:",
    "Thảo luận Bản mẫu:",
    "Thành viên:",
    "MediaWiki:",
    "Trợ giúp:",
    "Cổng thông tin:",
    "Dự án:",
    "Module:",
)


def clean(text):
    if not isinstance(text, str):
        return ""

    try:
        wikicode = mwparserfromhell.parse(text)
        text = wikicode.strip_code()

        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        return text

    except Exception:
        return ""


def process_csv(file_path):
    df = pd.read_csv(file_path)

    print("Initial:", len(df))

    df["title"] = df["title"].fillna("")
    df["text"] = df["text"].fillna("")

    # bỏ namespace không chứa tri thức
    df = df[~df["title"].str.startswith(BAD_PREFIXES, na=False)]

    # bỏ trang chính
    df = df[df["title"] != "Trang Chính"]

    # bỏ trang định hướng / giải nghĩa
    df = df[
        ~df["title"].str.contains(
            r"định hướng|giải nghĩa",
            case=False,
            na=False,
        )
    ]

    # bỏ redirect
    df = df[
        ~df["text"].str.match(
            r"(?i)^#redirect",
            na=False,
        )
    ]

    print("After filtering:", len(df))

    print("Starting cleaning...")
    df["text"] = df["text"].progress_apply(clean)

    # bỏ bài rỗng
    df = df[df["text"].str.len() > 0]

    # bỏ bài quá ngắn
    df = df[df["text"].str.len() > 800]

    print("Final:", len(df))

    df.to_csv(
        "cleaned.csv",
        index=False,
        encoding="utf-8",
    )


process_csv(r"E:\ProjectAI\Agent\data\clean\viwiki.csv")

Initial: 4534215
After filtering: 1606209
Starting cleaning...


Cleaning Wiki Text: 100%|█████████████████████████████████████████████████| 1606209/1606209 [1:09:34<00:00, 384.79it/s]


Final: 239629
